# OncoSeg — Train & Prove (real data, honest statistics)

> 🌐 English. This notebook **trains OncoSeg on real MSD Brain-Tumour MRI on a GPU** and rigorously tests the project's *defensible* claims — nothing is rigged to win.

## What this notebook pre-registers (before looking at results)

1. **H1 — Equivalence.** Is OncoSeg's segmentation accuracy *statistically equivalent* to a standard 3D U-Net? Test: paired **TOST** on per-subject mean Dice, margin **±0.02** (clinically justified: BraTS inter-rater Dice varies ~0.02–0.10, so a smaller model-vs-model gap is clinically indistinguishable).
2. **Efficiency.** Does that equivalence hold at **fewer parameters**? (measured from the trained checkpoints.)
3. **Cross-attention ablation.** What does the cross-attention skip actually contribute? Train `oncoseg` vs `oncoseg_no_xattn` (identical net, additive skips instead) and compare — **honestly, even if it does not help.**

**NOT claimed:** *superiority.* The pilot effect size is tiny (dz≈0.05 → ~3,700 subjects needed for 80% power), so 'OncoSeg is more accurate' is not testable here and we do not assert it. The honest headline is **equal accuracy at a smaller model**.

**Disclosed caveats (read before trusting any number):** single training seed unless you loop it; the U-Net that `train_all.py` trains is the **4.75M** 4-level config (so the honest ratio is ~1.6× vs OncoSeg's ~2.9M — the 19.2M 5-level U-Net is *not* trained here); the checkpoint is selected on the same 96-subject val set it is scored on (val==test — disclosed); calibration is poor on tumour voxels (foreground ECE≈0.49).

**Before running:** `Runtime → Change runtime type → GPU`, then `Runtime → Run all`.

## 1 · Setup — clone, install, one-time kernel restart

In [ ]:
import os
REPO = "https://github.com/danielchen26/OncoSeg-3D-Multi-Scale-Tumor-Segmentation-for-Automated-Treatment-Response-Assessment.git"
BRANCH = "fix/review-findings"
FLAG = "/content/.oncoseg_train_installed"   # file flag survives the kernel restart
if not os.path.isdir("/content/oncoseg"):
    !git clone --branch $BRANCH --depth 1 $REPO /content/oncoseg
%cd /content/oncoseg
!git log --oneline -1
if not os.path.exists(FLAG):
    !pip -q install -e "/content/oncoseg[dev]"
    open(FLAG, "w").close()
    print("\n>>> Installed. Restarting the kernel ONCE — then Runtime ▸ Run all again. <<<")
    import time; time.sleep(1); os.kill(os.getpid(), 9)
else:
    print("Dependencies ready — continuing.")

## 2 · GPU + import check

In [ ]:
import os; os.chdir("/content/oncoseg") if os.path.isdir("/content/oncoseg") else None
import torch, sys
print("Python", sys.version.split()[0], "| torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (training will be slow)")
for m in ["monai","nibabel","scipy","numpy"]:
    try: __import__(m); print(f"  {m}: OK")
    except Exception as e: print(f"  {m}: MISSING ({e})")

## 3 · Measure the real parameter counts (no training needed)

Resolves the documentation's parameter inconsistency directly from the code, before any claim.

In [ ]:
import sys, os; sys.path.insert(0, "/content/oncoseg" if os.path.isdir("/content/oncoseg") else os.getcwd())
from train_all import build_model
def npar(m): return sum(p.numel() for p in m.parameters())
roi=(96,96,96)
for name in ["oncoseg","unet3d","oncoseg_no_xattn"]:
    n = npar(build_model(name, roi, embed_dim=24))
    print(f"  {name:20s}: {n:>10,}  ({n/1e6:.2f}M)")
print("\nHonest ratio: OncoSeg (~2.9M) vs the UNet3D that train_all.py trains (4-level, ~4.75M) = ~1.6x fewer.")
print("A standard 5-level UNet3D is ~19.2M (OncoSeg ~6.7x smaller) but is NOT trained here.")

## 4 · Download the real dataset (~7 GB, once)

MSD Task01_BrainTumour — real clinical multi-modal brain MRI, 484 subjects, 388/96 train/val split (seed 42).

In [ ]:
!python data/scripts/download_msd.py --output data/raw
!python data/scripts/download_msd.py --output data/raw --verify-only

## 5 · Smoke test — prove the training pipeline runs (2 epochs, ~minutes)

**Not a result** — just confirms all three models train end-to-end before you commit hours. Skip to §6 for the real run.

In [ ]:
# Quick 2-epoch sanity run for all three models (outputs land in experiments/local_results/)
!python train_all.py --models oncoseg unet3d oncoseg_no_xattn --epochs 2 --roi-size 96 --embed-dim 24 --batch-size 1 --device cuda
print("\nSmoke run done — if this completed without error, the real run below will work.")

## 6 · Real training — 50 epochs, three models, identical settings

Same epochs / optimizer / schedule / augmentation / ROI for all three (fair comparison). Hours on a T4. Auto-resumes from `{name}_checkpoint.pth` if the session disconnects — just re-run. **Single seed** (42); loop this cell over seeds for a multi-seed result (disclosed limitation).

In [ ]:
!python train_all.py --models oncoseg unet3d oncoseg_no_xattn --epochs 50 --roi-size 96 --embed-dim 24 --batch-size 1 --val-interval 5 --device cuda

# Fairness gate: confirm ALL THREE reached 50 epochs (no OOM truncation)
import json, glob
for h in sorted(glob.glob("experiments/local_results/*_history.json")):
    d = json.load(open(h)); ep = len(d.get("val_dice_mean", d.get("train_loss", [])))
    print(f"  {os.path.basename(h):32s}: {ep} val/loss points, best_dice={d.get('best_dice','?')}")

## 7 · Evaluate all three models on the SAME val split (per-subject Dice)

`evaluate_checkpoint.py` only handles OncoSeg, so we evaluate inline: rebuild each model, load its `best.pth`, run the identical sliding-window inference on the identical validation subjects, and save per-subject Dice `[n,3]` arrays (region order [TC,WT,ET]) — the input the statistics need.

In [ ]:
import numpy as np, torch, os, sys
sys.path.insert(0, "/content/oncoseg" if os.path.isdir("/content/oncoseg") else os.getcwd())
from monai.data import DataLoader, Dataset
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
import train_all as T

roi=(96,96,96); device="cuda" if torch.cuda.is_available() else "cpu"
# reuse the EXACT training val split (build_data seed=42, val_split=0.2)
_, val_ds, n_tr, n_val = T.build_data(roi)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
print(f"val subjects: {n_val} (train {n_tr})")

def eval_model(name):
    m = T.build_model(name, roi, embed_dim=24).to(device).eval()
    ck = torch.load(f"experiments/local_results/{name}_best.pth", map_location=device, weights_only=False)
    m.load_state_dict(ck["model_state_dict"] if "model_state_dict" in ck else ck)
    dm = DiceMetric(include_background=True, reduction="none")
    rows=[]
    with torch.no_grad():
        for b in val_loader:
            x=b["image"].to(device); y=b["label"].to(device)
            logits = sliding_window_inference(x, roi, 1, lambda t: m(t)["pred"], overlap=0.25)
            pred = (torch.sigmoid(logits) > 0.5).float()
            dm.reset(); dm(y_pred=pred, y=y)
            rows.append(dm.aggregate().cpu().numpy()[0])
    arr=np.array(rows)  # [n,3] = [TC,WT,ET]
    np.save(f"experiments/local_results/{name}_per_subject_dice.npy", arr)
    print(f"  {name:20s}: per-subject Dice {arr.shape}, mean={np.nanmean(arr):.4f}")
    return arr

for name in ["oncoseg","unet3d","oncoseg_no_xattn"]:
    eval_model(name)

## 8 · The statistics (verified analysis code)

This `paired_equivalence()` was **verified on the committed pilot data** (it reproduces the known diff=+0.0025, TOST ±0.02 p=0.0009, Wilcoxon two-sided p=0.815) before this notebook was written — so the analysis you see is trustworthy; only the *inputs* change to your freshly-trained models.

In [ ]:
import numpy as np
from scipy import stats

def paired_equivalence(onc, unet, margins=(0.02,0.01), n_boot=20000, seed=0):
    """onc, unet: [n_subjects] paired per-subject mean-Dice vectors. Returns the full honest stat report."""
    m = ~(np.isnan(onc)|np.isnan(unet)); a,b = onc[m], unet[m]; d = a-b; n=len(d)
    out = {"n":n, "mean_onc":float(a.mean()), "mean_unet":float(b.mean()), "mean_diff":float(d.mean()),
           "sd_diff":float(d.std(ddof=1)), "cohens_dz":float(d.mean()/d.std(ddof=1))}
    # bootstrap CI on mean diff
    rng=np.random.default_rng(seed)
    boot=np.array([d[rng.integers(0,n,n)].mean() for _ in range(n_boot)])
    out["ci95"]=[float(np.percentile(boot,2.5)), float(np.percentile(boot,97.5))]
    # two-sided Wilcoxon (superiority test — report honestly)
    try: out["wilcoxon_two_sided_p"]=float(stats.wilcoxon(a,b).pvalue)
    except ValueError: out["wilcoxon_two_sided_p"]=1.0
    out["onc_wins"]=int((a>b).sum()); out["unet_wins"]=int((b>a).sum())
    # TOST equivalence at each margin
    se=d.std(ddof=1)/np.sqrt(n)
    out["tost"]={}
    for delta in margins:
        t1=(d.mean()-(-delta))/se; p1=stats.t.sf(t1,n-1)
        t2=(delta-d.mean())/se; p2=stats.t.sf(t2,n-1)
        p=max(p1,p2)
        out["tost"][f"±{delta}"]={"p":float(p),"equivalent":bool(p<0.05)}
    return out

## 9 · Verdict 1 — OncoSeg vs UNet3D (equivalence + efficiency)

In [ ]:
import numpy as np
o = np.nanmean(np.load("experiments/local_results/oncoseg_per_subject_dice.npy"), axis=1)
u = np.nanmean(np.load("experiments/local_results/unet3d_per_subject_dice.npy"), axis=1)
r = paired_equivalence(o, u)
print("=== OncoSeg vs UNet3D ===")
print(f"  mean Dice: OncoSeg {r['mean_onc']:.4f}  vs UNet3D {r['mean_unet']:.4f}  (diff {r['mean_diff']:+.4f})")
print(f"  95% CI on diff: [{r['ci95'][0]:+.4f}, {r['ci95'][1]:+.4f}]   Cohen dz={r['cohens_dz']:.3f}")
print(f"  Wilcoxon two-sided p={r['wilcoxon_two_sided_p']:.4f}  (OncoSeg wins {r['onc_wins']}, UNet3D {r['unet_wins']})")
for k,v in r["tost"].items(): print(f"  TOST {k}: p={v['p']:.4f} -> {'EQUIVALENT' if v['equivalent'] else 'not established'}")
eq = r["tost"].get("±0.02",{}).get("equivalent")
print("\nVERDICT:", "statistically EQUIVALENT within ±0.02 Dice" if eq else "equivalence NOT established at ±0.02")
print("       (superiority is NOT claimed — see pre-registration)")

## 10 · Verdict 2 — the cross-attention ablation (what it actually buys)

OncoSeg (cross-attention skips) vs `oncoseg_no_xattn` (additive skips). This is the first real evidence for whether the architectural novelty helps. **Interpreted honestly:** a positive, non-trivial gap supports the design; a tie means cross-attention is not the source of any benefit and should be reported as such.

In [ ]:
import numpy as np
o  = np.nanmean(np.load("experiments/local_results/oncoseg_per_subject_dice.npy"), axis=1)
nx = np.nanmean(np.load("experiments/local_results/oncoseg_no_xattn_per_subject_dice.npy"), axis=1)
r = paired_equivalence(o, nx)
print("=== OncoSeg (cross-attn) vs oncoseg_no_xattn (additive skip) ===")
print(f"  mean Dice: xattn {r['mean_onc']:.4f}  vs no-xattn {r['mean_unet']:.4f}  (diff {r['mean_diff']:+.4f})")
print(f"  95% CI on diff: [{r['ci95'][0]:+.4f}, {r['ci95'][1]:+.4f}]   Cohen dz={r['cohens_dz']:.3f}")
print(f"  Wilcoxon two-sided p={r['wilcoxon_two_sided_p']:.4f}  (xattn wins {r['onc_wins']}, no-xattn {r['unet_wins']})")
sig = r["wilcoxon_two_sided_p"] < 0.05
print("\nVERDICT:", ("cross-attention gives a statistically detectable Dice difference (%+.4f)" % r["mean_diff"]) if sig
      else "no statistically detectable Dice difference from cross-attention on this run — report honestly (it adds ~0.14M params for no measured Dice gain here).")

## 11 · Honest conclusion (auto-filled)

Fill the summary from the numbers this run actually produced — and keep the refusals honest.

In [ ]:
import numpy as np
o=np.nanmean(np.load("experiments/local_results/oncoseg_per_subject_dice.npy"),axis=1)
u=np.nanmean(np.load("experiments/local_results/unet3d_per_subject_dice.npy"),axis=1)
nx=np.nanmean(np.load("experiments/local_results/oncoseg_no_xattn_per_subject_dice.npy"),axis=1)
from train_all import build_model
par=lambda n: sum(p.numel() for p in build_model(n,(96,96,96),embed_dim=24).parameters())
ru=paired_equivalence(o,u); rx=paired_equivalence(o,nx)
print("="*64)
print("HONEST SUMMARY (this run)")
print("="*64)
print(f"OncoSeg {par('oncoseg')/1e6:.2f}M  vs  UNet3D {par('unet3d')/1e6:.2f}M  ({par('unet3d')/par('oncoseg'):.2f}x fewer params)")
print(f"Accuracy: mean Dice {ru['mean_onc']:.4f} vs {ru['mean_unet']:.4f}; TOST±0.02 {'EQUIVALENT' if ru['tost']['±0.02']['equivalent'] else 'not established'}")
print(f"Cross-attention: xattn {rx['mean_onc']:.4f} vs no-xattn {rx['mean_unet']:.4f} (diff {rx['mean_diff']:+.4f}, Wilcoxon p={rx['wilcoxon_two_sided_p']:.3f})")
print("\nCAN claim: equal accuracy at fewer params (if TOST equivalent); the measured cross-attention effect above.")
print("CANNOT claim: accuracy superiority (underpowered); well-calibrated (fg ECE~0.49); the 5-level 19.2M ratio (not trained); real longitudinal validation.")
print("LIMITATIONS: single seed; val==test checkpoint selection; one dataset (MSD Task01).")